In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import DataLoader, Dataset

##Uploading data

In [ ]:
df = pd.read_csv('/content/Yelp Restaurant Reviews.csv')
df.head()

,Yelp URL,Rating,Date,Review Text
0,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5.0,1/22/2022,All I can say is they have very good ice cream...
1,https://www.yelp.com/biz/sidney-dairy-barn-sidney,4.0,6/26/2022,Nice little local place for ice cream.My favor...
2,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5.0,8/7/2021,A delicious treat on a hot day! Staff was very...
3,https://www.yelp.com/biz/sidney-dairy-barn-sidney,4.0,7/28/2016,This was great service and a fun crew! I got t...
4,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5.0,6/23/2015,This is one of my favorite places to get ice c...


In [ ]:
df = df.drop(['Yelp URL', "Date"], axis = 1)
df = df[df['Rating'] != 3]
df.sample(10)

,Rating,Review Text
3984,4.0,"Yes, this place is great. A little overrated b..."
3629,4.0,Boy and I stopped by after an afternoon stroll...
2737,1.0,Not only did they yell at everyone as they wal...
2127,5.0,My favorite ice cream ever!!!! The only proble...
5506,5.0,I invented a new name for the macaroons here. ...
2349,5.0,You can't go wrong here. There are a ton of ch...
7652,5.0,YUM to all things creative and delicious!!! I'...
541,1.0,Italian bakery with no cannoli & tiramisu? Wha...
3541,5.0,"Love the breakfast sandwich with egg, spinach,..."
59,5.0,I was really impressed with my whole experienc...


##Data preprocessing

In [ ]:
df['Rating'].value_counts()

,count
Rating,
5.0,7529
4.0,3322
2.0,1064
1.0,992


In [ ]:
diction = {5:'pos',4:'pos',1:'neg',2:'neg'}
df['Rating'] = df['Rating'].map(diction)

In [ ]:
df['Rating'].value_counts()

,count
Rating,
pos,10851
neg,2056


In [ ]:
df_pos = df[df['Rating'] == 'pos'].sample(750, random_state = 42)
df_neg = df[df['Rating'] == 'neg'].sample(750, random_state = 42)
df = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=42) \
       .reset_index(drop=True)
df.shape

(1500, 2)

##Tokenizer and Encoding

In [ ]:
def tokenize(text):
  return text.lower().split()

counter = Counter(word for text in df['Review Text'] for word in tokenize(text))
most_common = counter.most_common(5000)
vocab = {word:i+2 for i, (word,_) in enumerate(most_common)}

vocab['<pad>'] = 0
vocab['<unk>'] = 1

def encode(text):
  return torch.tensor([vocab.get(word, 1) for word in tokenize(text)])

In [ ]:
max(len(x) for x in [encode(text) for text in df['Review Text']])

750

##DataLoader

In [ ]:
class StarsDataset(Dataset):
  def __init__(self,df):
    self.X = [encode(text) for text in df['Review Text']]
    self.y = [torch.tensor([1.0 if s == 'pos' else 0.0]) for s in df['Rating']]

  def __len__(self):
    return len(self.X)

  def __getitem__(self, index):
    return self.X[index], self.y[index]

def collate_fn(batch):
  Xs, ys = zip(*batch)
  max_len = max(len(x) for x in Xs)
  padded_X = [torch.cat([x, torch.zeros(max_len - len(x), dtype=torch.long)]) for x in Xs]
  return torch.stack(padded_X), torch.stack(ys)

train_loader = DataLoader(StarsDataset(df), collate_fn=collate_fn, batch_size=32, shuffle=True)

##Model

In [ ]:
class SentimentRNN(nn.Module):
  def __init__(self, vocab_size, embedding_dim = 100, hidden_dim = 256):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
    self.fc = nn.Linear(hidden_dim, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.embedding(x)
    _, h = self.rnn(x)
    x = self.fc(h.squeeze(0))
    return self.sigmoid(x)

In [ ]:
import torch

print(torch.cuda.is_available())

True


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentimentRNN(len(vocab)).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr= 0.001)

##Training

In [ ]:
for epoch in range(100):
  total_loss = 0
  model.train()
  for X, y in train_loader:
    X = X.to(device)
    y = y.to(device)
    y_pred = model(X)
    loss = criterion(y_pred,y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  if epoch%10 == 0:
      print(f'Epoch: {epoch}, Loss: {total_loss/len(train_loader):.4f}')

Epoch: 0, Loss: 0.6976
Epoch: 10, Loss: 0.6984
Epoch: 20, Loss: 0.6941
Epoch: 30, Loss: 0.6913
Epoch: 40, Loss: 0.6912
Epoch: 50, Loss: 0.6884
Epoch: 60, Loss: 0.7026
Epoch: 70, Loss: 0.6964
Epoch: 80, Loss: 0.6909
Epoch: 90, Loss: 0.6868


##Evaluation

In [ ]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        pred = (model(X) > 0.5).float()

        correct += (pred == y).sum().item()
        total += y.size(0)

print(correct / total)

0.5186666666666667


In [ ]:
def predict(text):
  model.eval()
  with torch.no_grad():
    x = encode(text)
    x = x[:100]
    if len(x) < 100:
      x = torch.cat([x, torch.zeros(100-len(x), dtype = torch.long)])
    x = x.unsqueeze(0)
    x = x.to(device) # Ensure the input tensor is on the correct device
    y_pred = model(x)

    prob = y_pred.item()
    label = 'positive' if prob > 0.5 else 'negative'
    print(f'Prob: {prob:.2f}, Label: {label}')

In [ ]:
predict('The food is delicious, but the service is very slow')
predict('I will never come back to this restaurant. The waiter was very rude')

Prob: 0.47, Label: negative
Prob: 0.47, Label: negative
